In [1]:
import os
import random
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# --- PARAMETERS ---
frame_rate = 5  # Extract every nth frame
image_size = (224, 224)  # Input size for CNN
batch_size = 32
epochs = 50
train_test_ratio = 0.8  # Train-test split ratio


In [2]:
# --- STEP 1: SPLIT VIDEOS ---
def split_videos(videos_folder, split_ratio=0.8):
    """
    Splits the videos in the given folder into training and testing sets.
    """
    video_files = [os.path.join(videos_folder, f) for f in os.listdir(videos_folder) 
                   if os.path.isfile(os.path.join(videos_folder, f)) and f.endswith('.mp4')]
    
    random.shuffle(video_files)
    split_index = int(len(video_files) * split_ratio)
    train_videos = video_files[:split_index]
    test_videos = video_files[split_index:]
    
    return train_videos, test_videos

videos_folder = "./videos"  # Update this path to your videos folder
train_videos, test_videos = split_videos(videos_folder)

print(f"Training videos: {len(train_videos)}, Testing videos: {len(test_videos)}")
print("Example training video:", train_videos[0] if train_videos else "None")
print("Example testing video:", test_videos[0] if test_videos else "None")

Training videos: 9584, Testing videos: 2396
Example training video: ./videos\62690.mp4
Example testing video: ./videos\44206.mp4


In [3]:
import cv2
import os

def extract_frames(video_folder, output_folder, frame_rate=1):
    os.makedirs(output_folder, exist_ok=True)
    
    for category in os.listdir(video_folder):
        category_path = os.path.join(video_folder, category)
        output_category_path = os.path.join(output_folder, category)
        os.makedirs(output_category_path, exist_ok=True)
        
        if os.path.isdir(category_path):
            for video_file in os.listdir(category_path):
                video_path = os.path.join(category_path, video_file)
                cap = cv2.VideoCapture(video_path)
                
                frame_count = 0
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break
                    if frame_count % frame_rate == 0:  # Extract frames at specified frame rate
                        frame_file = os.path.join(output_category_path, f"{video_file}_frame{frame_count}.jpg")
                        cv2.imwrite(frame_file, frame)
                    frame_count += 1
                
                cap.release()
                
    print("Frames extracted successfully!")

# Example usage
extract_frames("videos", "frames_output", frame_rate=5)  # Extract a frame every 5th frame

Frames extracted successfully!


In [4]:
import cv2
import numpy as np

def preprocess_video(video_path, frame_rate=5):
    """
    Extract and preprocess frames from a video file.

    Parameters:
        video_path (str): Path to the video file.
        frame_rate (int): Extract every nth frame.

    Returns:
        frames (list): List of preprocessed frames.
    """
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_rate == 0:  # Extract every nth frame
            frame = cv2.resize(frame, (224, 224))  # Resize to fit the model
            frame = frame.astype(np.float32) / 255.0  # Normalize pixel values
            frames.append(frame)
        frame_count += 1

    cap.release()
    return np.array(frames)

# Example usage
train_data = []
train_labels = []

for i, video in enumerate(train_videos[:50]):  # Limit to 50 for quick testing
    frames = preprocess_video(video)
    train_data.extend(frames)
    train_labels.extend([i] * len(frames))  # Assign labels (update logic as needed)

train_data = np.array(train_data)
train_labels = np.array(train_labels)

print(f"Processed {len(train_data)} frames for training.")

Processed 663 frames for training.


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define CNN Model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(len(set(train_labels)), activation='softmax')  # Number of classes
])

# Compile the Model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the Model
history = model.fit(
    train_data,
    train_labels,
    batch_size=32,
    epochs=25,
    validation_split=0.2  # Reserve 20% of training data for validation
)

# Save the Model
model.save("sign_language_model.h5")
print("Model saved successfully!")

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 618ms/step - accuracy: 0.0208 - loss: 4.1287 - val_accuracy: 0.0000e+00 - val_loss: 4.1245
Epoch 2/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 604ms/step - accuracy: 0.0536 - loss: 3.7562 - val_accuracy: 0.0000e+00 - val_loss: 4.7374
Epoch 3/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 592ms/step - accuracy: 0.0967 - loss: 3.2828 - val_accuracy: 0.0000e+00 - val_loss: 6.7316
Epoch 4/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 595ms/step - accuracy: 0.2252 - loss: 2.8024 - val_accuracy: 0.0000e+00 - val_loss: 7.0744
Epoch 5/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 603ms/step - accuracy: 0.3567 - loss: 2.1681 - val_accuracy: 0.0000e+00 - val_loss: 8.6753
Epoch 6/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 583ms/step - accuracy: 0.4315 - loss: 1.8857 - val_accuracy: 0.0000e+00 - val_loss: 7.7990
Epoch 7/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 613ms/step - accuracy: 0.5174 - loss: 1.5821 - val_accuracy: 0.0000e+00 - val_loss: 8.9914
Epoch 8/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 603ms/step - accuracy: 0.57

Model saved successfully!


In [13]:
# Preprocess and prepare test data
test_data = []
test_labels = []

for i, video in enumerate(test_videos[:20]):  # Process a subset for quick testing
    frames = preprocess_video(video)
    test_data.extend(frames)
    test_labels.extend([i] * len(frames))  # Adjust labels based on your classification logic

test_data = np.array(test_data)
test_labels = np.array(test_labels)

# Evaluate the model on the test dataset
loss, accuracy = model.evaluate(test_data, test_labels)
print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step - accuracy: 0.1298 - loss: 7.4408
Test Loss: 7.426368713378906, Test Accuracy: 0.11397058516740799
